### **Aprimoramento de imagens com baixa luminosidade utilizando aprendizado profundo para aplicações de navegação de robôs móveis**

### **Pré processamento**

In [3]:
import os
import gc
import random

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf

from os import listdir
from os.path import isfile, join
from skimage.metrics import structural_similarity
from tensorflow.keras import Input, layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K

# Configurações gerais

TRAIN_LOW_PATH = '/content/drive/MyDrive/DATASETS/Real_captured/Train/Low'
TRAIN_NORMAL_PATH = '/content/drive/MyDrive/DATASETS/Real_captured/Train/Normal'
TEST_LOW_PATH = 'Real_captured/Test/Low'
TEST_NORMAL_PATH = 'Real_captured/Test/Normal'

MODEL_PATH = 'melhor_modelo.h5'

IMAGE_WIDTH = 600
IMAGE_HEIGHT = 400


# # Montagem do Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
def load_image_folder(folder_path, width=IMAGE_WIDTH, height=IMAGE_HEIGHT):
    """Carrega, ordena, redimensiona e converte imagens BGR para RGB."""
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f'Diretório não encontrado: {folder_path}')

    image_files = sorted(
        file_name for file_name in listdir(folder_path)
        if isfile(join(folder_path, file_name))
    )

    images = []
    for file_name in image_files:
        image_path = join(folder_path, file_name)
        image = cv2.imread(image_path, cv2.IMREAD_COLOR)

        if image is None:
            raise ValueError(f'Não foi possível ler a imagem: {image_path}')

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (width, height))
        images.append(image)

    if not images:
        raise ValueError(f'Nenhuma imagem encontrada em: {folder_path}')

    return np.asarray(images)


def read_images_train():
    print('Carregando imagens de treinamento...')
    X_train = load_image_folder(TRAIN_LOW_PATH)
    Y_train = load_image_folder(TRAIN_NORMAL_PATH)

    if len(X_train) != len(Y_train):
        raise ValueError('As pastas Low e Normal possuem quantidades diferentes de imagens.')

    print('X_train:', X_train.shape)
    print('Y_train:', Y_train.shape)
    return X_train, Y_train


def read_images_test():
    print('Carregando imagens de teste...')
    X_test = load_image_folder(TEST_LOW_PATH)
    Y_test = load_image_folder(TEST_NORMAL_PATH)

    if len(X_test) != len(Y_test):
        raise ValueError('As pastas de teste Low e Normal possuem quantidades diferentes de imagens.')

    print('X_test:', X_test.shape)
    print('Y_test:', Y_test.shape)
    return X_test, Y_test


def prep_pixels(train, test):
    print('Normalizando imagens...')
    train_norm = train.astype(np.float32) / 255.0
    test_norm = test.astype(np.float32) / 255.0
    return train_norm, test_norm

### **Arquitetura**

In [5]:
mse_fn = MeanSquaredError()


def ssim_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))


def psnr_loss(y_true, y_pred):
    return -tf.reduce_mean(tf.image.psnr(y_true, y_pred, max_val=1.0))


def combined_loss(y_true, y_pred):
    mse = mse_fn(y_true, y_pred)
    ssim_value = ssim_loss(y_true, y_pred)
    return 0.73 * mse + 0.27 * ssim_value


def self_attention_module(x):
    num_filters = x.shape[-1]
    attention = layers.Conv2D(num_filters, kernel_size=1, padding='same')(x)
    attention = layers.BatchNormalization()(attention)
    attention = layers.Activation('relu')(attention)
    attention = layers.Conv2D(num_filters, kernel_size=1, padding='same')(attention)
    attention = layers.Activation('sigmoid')(attention)
    return layers.Multiply()([x, attention])


def match_shape(to_adjust, target):
    height_diff = target.shape[1] - to_adjust.shape[1]
    width_diff = target.shape[2] - to_adjust.shape[2]

    # Ajusta primeiro por recorte e depois por preenchimento.
    crop_height = max(-height_diff, 0)
    crop_width = max(-width_diff, 0)

    if crop_height or crop_width:
        crop_top = crop_height // 2
        crop_bottom = crop_height - crop_top
        crop_left = crop_width // 2
        crop_right = crop_width - crop_left
        to_adjust = layers.Cropping2D(((crop_top, crop_bottom), (crop_left, crop_right)))(to_adjust)

    pad_height = max(height_diff, 0)
    pad_width = max(width_diff, 0)

    if pad_height or pad_width:
        pad_top = pad_height // 2
        pad_bottom = pad_height - pad_top
        pad_left = pad_width // 2
        pad_right = pad_width - pad_left
        to_adjust = layers.ZeroPadding2D(((pad_top, pad_bottom), (pad_left, pad_right)))(to_adjust)

    return to_adjust

def LOL_encoder(input_shape, filters, dropout_rate, learning_rate, kernel_sizes):
    if isinstance(filters, int):
        filters = [filters] * 9
    if isinstance(kernel_sizes, int):
        kernel_sizes = [kernel_sizes] * 9

    if len(filters) != 9 or len(kernel_sizes) != 9:
        raise ValueError('filters e kernel_sizes devem possuir 9 elementos.')

    inputs = Input(shape=input_shape)

    # Encoder
    x1 = layers.Conv2D(filters[0], (kernel_sizes[0], kernel_sizes[0]), padding='same', kernel_initializer='he_normal', bias_initializer='zeros')(inputs)
    x1 = layers.Activation('relu')(layers.BatchNormalization()(x1))
    p1 = layers.MaxPooling2D((2, 2))(x1)

    x2 = layers.Conv2D(filters[1], (kernel_sizes[1], kernel_sizes[1]), padding='same', kernel_initializer='he_normal', bias_initializer='zeros')(p1)
    x2 = layers.Activation('relu')(layers.BatchNormalization()(x2))
    p2 = layers.MaxPooling2D((2, 2))(x2)

    x3 = layers.Conv2D(filters[2], (kernel_sizes[2], kernel_sizes[2]), padding='same', kernel_initializer='he_normal', bias_initializer='zeros')(p2)
    x3 = layers.Activation('relu')(layers.BatchNormalization()(x3))
    p3 = layers.MaxPooling2D((2, 2))(x3)

    x4 = layers.Conv2D(filters[3], (kernel_sizes[3], kernel_sizes[3]), padding='same', kernel_initializer='he_normal', bias_initializer='zeros')(p3)
    x4 = layers.Activation('relu')(layers.BatchNormalization()(x4))
    p4 = layers.MaxPooling2D((2, 2))(x4)

    x5 = layers.Conv2D(filters[4], (kernel_sizes[4], kernel_sizes[4]), padding='same', kernel_initializer='he_normal', bias_initializer='zeros')(p4)
    x5 = layers.Activation('relu')(layers.BatchNormalization()(x5))
    p5 = layers.MaxPooling2D((2, 2))(x5)

    # Bottleneck com atenção
    bottleneck = layers.Conv2D(filters[5], (kernel_sizes[5], kernel_sizes[5]), padding='same', kernel_initializer='he_normal', bias_initializer='zeros')(p5)
    bottleneck = layers.Activation('relu')(layers.BatchNormalization()(bottleneck))
    bottleneck = self_attention_module(bottleneck)

    # Decoder
    decoder_specs = [
        (filters[6], kernel_sizes[6], x5),
        (filters[7], kernel_sizes[7], x4),
        (filters[8], kernel_sizes[8], x3),
        (64, 3, x2),
        (64, 3, x1),
    ]

    x = bottleneck
    
    for index, (num_filters, kernel_size, skip) in enumerate(decoder_specs):
        x = layers.Conv2DTranspose(
            num_filters, (kernel_size, kernel_size),
            strides=(2, 2),
            padding='same',
            kernel_initializer='he_normal',
            bias_initializer='zeros'
        )(x)
        x = match_shape(x, skip)
        x = layers.Concatenate()([x, skip])
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)

        if index < 3:
            x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same', kernel_initializer='he_normal', bias_initializer='zeros')(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=Adam(learning_rate),
        loss=combined_loss,
        metrics=[
            MeanSquaredError(name='mse'),
            tf.keras.metrics.MeanMetricWrapper(ssim_loss, name='ssim')
        ]
    )
    return model

### **Treinamento**

In [6]:
def calculate_psnr(image_a, image_b):
    image_a = (image_a * 255).clip(0, 255).astype(np.uint8)
    image_b = (image_b * 255).clip(0, 255).astype(np.uint8)
    return cv2.PSNR(image_a, image_b)


def calculate_ssim(image_a, image_b):
    channels = [
        structural_similarity(image_a[:, :, channel], image_b[:, :, channel], data_range=1.0)
        for channel in range(3)
    ]
    return float(np.mean(channels))


def evaluate_model(model, X_test, Y_test):
    ssim_scores, psnr_scores, loss_scores = [], [], []

    for input_image, target_image in zip(X_test, Y_test):
        prediction = model.predict(np.expand_dims(input_image, axis=0), verbose=0)[0]
        ssim_scores.append(calculate_ssim(prediction, target_image))
        psnr_scores.append(calculate_psnr(target_image, prediction))
        loss_scores.append(np.mean((prediction - target_image) ** 2))

    return np.mean(ssim_scores), np.mean(loss_scores), np.mean(psnr_scores)


def save_parameters(best_params, ssim_score, psnr_score, best_weights, iteration, path='new_best.csv'):
    row = {
        'iteration': iteration,
        'filters': str(best_params['filters']),
        'dropout': best_params['dropout'],
        'learning_rate': best_params['learning_rate'],
        'kernel_sizes': str(best_params['kernel_sizes']),
        'batch_size': best_params['batch_size'],
        'epoch': best_params['epoch'],
        'loss': best_params.get('loss'),
        'ssim': ssim_score,
        'psnr': psnr_score,
        'loss_weights': str(best_weights)
    }
    pd.DataFrame([row]).to_csv(path, mode='a', header=not os.path.isfile(path), index=False)


def train_model(X_train, Y_train, input_shape, params):
    model = LOL_encoder(
        input_shape,
        filters=params['filters'],
        dropout_rate=params['dropout'],
        learning_rate=params['learning_rate'],
        kernel_sizes=params['kernel_sizes']
    )

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    history = model.fit(
        X_train, Y_train,
        epochs=params['epoch'],
        batch_size=params['batch_size'],
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )
    return model, history

### **Random search**

In [7]:
def normalize_weights(weights):
    weights = [max(weight, 0) for weight in weights]
    total = sum(weights)
    return [weight / total for weight in weights] if total else [1.0 / len(weights)] * len(weights)


def random_search(X_train, Y_train, input_shape, num_trials=15):
    print('Iniciando Random Search...')
    best_ssim = -1.0
    best_loss = float('inf')
    best_psnr = -1.0
    best_model = None
    best_params = None
    best_weights = None

    for trial in range(num_trials):
        K.clear_session()
        gc.collect()

        filters = [random.choice([96, 128, 256]) for _ in range(9)]
        dropout_rate = random.choice([0.1, 0.2])
        learning_rate = random.choice([1e-3, 5e-4, 1e-4])
        kernel_sizes = [random.choice([3, 5]) for _ in range(9)]
        batch_size = random.choice([1, 2])
        epochs = random.randint(30, 60)

        w_mse, w_ssim, w_psnr = normalize_weights([
            random.uniform(0.1, 0.7) for _ in range(3)
        ])

        def trial_loss(y_true, y_pred):
            mse = mse_fn(y_true, y_pred)
            return w_mse * mse + w_ssim * ssim_loss(y_true, y_pred) + w_psnr * psnr_loss(y_true, y_pred)

        params = {
            'filters': filters,
            'dropout': dropout_rate,
            'learning_rate': learning_rate,
            'kernel_sizes': kernel_sizes,
            'batch_size': batch_size,
            'epoch': epochs
        }

        print(f'\nTentativa {trial + 1}/{num_trials}')
        print(f'Filtros: {filters} | Dropout: {dropout_rate} | LR: {learning_rate}')
        print(f'Kernels: {kernel_sizes} | Batch: {batch_size} | Épocas: {epochs}')

        model = LOL_encoder(input_shape, filters, dropout_rate, learning_rate, kernel_sizes)
        model.compile(
            optimizer=Adam(learning_rate),
            loss=trial_loss,
            metrics=[
                MeanSquaredError(name='mse'),
                tf.keras.metrics.MeanMetricWrapper(ssim_loss, name='ssim'),
                tf.keras.metrics.MeanMetricWrapper(psnr_loss, name='psnr')
            ]
        )

        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        model.fit(
            X_train, Y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=0.2,
            callbacks=[early_stop],
            verbose=1
        )

        val_size = max(1, int(0.2 * len(X_train)))
        X_val, Y_val = X_train[-val_size:], Y_train[-val_size:]
        ssim_score, loss_score, psnr_score = evaluate_model(model, X_val, Y_val)

        print(f'Validação | Loss: {loss_score:.4f} | SSIM: {ssim_score:.4f} | PSNR: {psnr_score:.4f}')

        if ssim_score > best_ssim:
            best_ssim, best_loss, best_psnr = ssim_score, loss_score, psnr_score
            best_model = model
            best_params = {**params, 'loss': loss_score}
            best_weights = (w_mse, w_ssim, w_psnr)

    if best_model is None:
        raise RuntimeError('O Random Search não produziu um modelo.')

    print('\nMelhor modelo encontrado')
    print(f'SSIM: {best_ssim:.4f} | Loss: {best_loss:.4f} | PSNR: {best_psnr:.4f}')
    print('Parâmetros:', best_params)
    print('Pesos da loss:', best_weights)

    save_parameters(best_params, best_ssim, best_psnr, best_weights, num_trials)
    return best_model, best_ssim, best_loss, best_params

### **Treinamento principal**

In [8]:
# Parâmetros utilizados quando RANDOM_SEARCH = False
RANDOM_SEARCH = False

FIXED_PARAMS = {
    'filters': [256, 256, 256, 96, 256, 256, 96, 96, 256],
    'dropout': 0.2,
    'learning_rate': 0.0005,
    'kernel_sizes': [3, 5, 3, 5, 3, 5, 3, 5, 5],
    'batch_size': 2,
    'epoch': 39
}

# Verifica a GPU
gpu_name = tf.test.gpu_device_name()
if gpu_name != '/device:GPU:0':
    raise SystemError('GPU não encontrada.')
print(f'GPU detectada: {gpu_name}')

# Carrega e normaliza os dados
X_train, Y_train = read_images_train()
X_train, Y_train = prep_pixels(X_train, Y_train)
input_shape = X_train.shape[1:]

# Executa a busca ou utiliza os parâmetros definidos
if RANDOM_SEARCH:
    lol_model, best_ssim, best_loss, best_params = random_search(
        X_train, Y_train, input_shape, num_trials=15
    )
else:
    print('Random Search desativado. Utilizando parâmetros fixos.')
    best_params = FIXED_PARAMS
    lol_model, history = train_model(X_train, Y_train, input_shape, best_params)

# Salva o modelo treinado
lol_model.save(MODEL_PATH, include_optimizer=False)
print(f'Modelo salvo em: {MODEL_PATH}')


KeyboardInterrupt



### **TESTE**

In [ ]:
# Diretórios de teste e saída
TEST_OUTPUT_DIR = 'Results_lolv2'

os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

# Carrega o modelo salvo
test_model = tf.keras.models.load_model(MODEL_PATH, compile=False)

# Carrega as imagens de teste
X_test = load_image_folder(TEST_LOW_PATH)
Y_test = load_image_folder(TEST_NORMAL_PATH)

if len(X_test) != len(Y_test):
    raise ValueError('As pastas de teste Low e Normal possuem quantidades diferentes.')

# Avalia e salva as imagens aprimoradas
psnr_values, ssim_values = [], []

for index, (input_image, target_image) in enumerate(zip(X_test, Y_test)):
    input_norm = input_image.astype(np.float32) / 255.0
    prediction = test_model.predict(np.expand_dims(input_norm, axis=0), verbose=0)[0]

    output_rgb = (prediction * 255.0).clip(0, 255).astype(np.uint8)
    output_bgr = cv2.cvtColor(output_rgb, cv2.COLOR_RGB2BGR)
    target_bgr = cv2.cvtColor(target_image, cv2.COLOR_RGB2BGR)

    output_path = os.path.join(TEST_OUTPUT_DIR, f'result_{index:04d}.png')
    cv2.imwrite(output_path, output_bgr)

    psnr_value = cv2.PSNR(target_bgr, output_bgr)
    ssim_value = structural_similarity(target_bgr, output_bgr, channel_axis=-1, data_range=255)

    psnr_values.append(psnr_value)
    ssim_values.append(ssim_value)

    print(f'Imagem {index + 1}/{len(X_test)} | PSNR: {psnr_value:.2f} | SSIM: {ssim_value:.4f}')

print(f'\nPSNR médio: {np.mean(psnr_values):.4f}')
print(f'SSIM médio: {np.mean(ssim_values):.4f}')